# Mixture-of-Experts (MoE) decoder LM

**Paper:** Switch Transformers: Scaling to Trillion Parameter Models with Simple and Efficient Sparsity
**Difficulty:** hard · **Estimated time:** 14 hours

## Why this exercise
MoE is the architectural lever behind every frontier model: more params, same FLOPs. Building one teaches routing, top-k dispatch, and load balancing.

## Learning objective
Implement the full sparse MoE transformer LM and train it end-to-end with an auxiliary load-balancing loss.

## Context
A Switch/Mixtral-style block replaces the dense MLP with N experts and a router that sends each token to its top-k. Routed outputs are weighted by router probabilities. Without a balance loss the router collapses to one expert; with it, all experts get used.

## Prerequisites
- Softmax routing
- torch.topk and torch.scatter
- Auxiliary load-balancing loss

## Concept ladder
1. Router projects tokens to N expert logits and selects top-k
2. ExpertMLP is the same as a vanilla MLP block
3. SparseMoE dispatches each token to its k experts and weights outputs
4. Track router stats and add a balance loss to keep experts utilised

## Tasks
- Task 1: TopKRouter and ExpertMLP.
- Task 2: SparseMoE forward that dispatches tokens, runs each expert, and combines outputs.
- Task 3: MoETransformerBlock with attention + SparseMoE.
- Task 4: MoELM stacking blocks. train_one_epoch should optionally add a load-balancing loss term.

## Milestones
1. **Sparse forward**
   - Goal: MoELM returns (B, T, V) logits using only top-k experts per token.
   - Check: `test_moe_forward_shape passes.`
2. **Trains on one batch**
   - Goal: Cross-entropy on a fixed batch halves within 100 steps.
   - Check: `test_moe_overfits_one_batch passes.`
3. **Pipeline runs**
   - Goal: run_pipeline returns train_loss and val_loss.
   - Check: `test_pipeline_runs_and_reports_metrics passes.`

## Setup
CPU is enough. Synthetic next-token sequences.

## How to work
- Implement the TODOs in the code cell below.
- Run the cell; the included tests identify the first failing requirement.
- Keep the implementation in one file-equivalent cell unless the exercise says otherwise.
- Finish only when the end-to-end pipeline and all checks pass.

In [ ]:
# Colab setup — PyTorch is normally preinstalled.
try:
    import torch
except ImportError:
    %pip install -q torch

import torch
print('PyTorch:', torch.__version__)
print('Device:', 'cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# Project: Mixture-of-Experts (MoE) decoder LM from scratch
# Paper: Switch Transformers: Scaling to Trillion Parameter Models with Simple and Efficient Sparsity (Fedus, Zoph, Shazeer, 2021)
# Goal: implement the full sparse MoE transformer architecture AND the end-to-end pretraining pipeline with a load-balancing loss
# Compute: CPU is enough at the tiny config used by the tests
# Rule: keep this as ONE file.
# Run when done: python assignment.py

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset


# =========================
# STUDENT CODE STARTS HERE
# =========================


class TokenEmbedding(nn.Module):
    def __init__(self, vocab_size, embed_dim, max_seq_len):
        super().__init__()
        raise AssertionError("TODO: implement TokenEmbedding.__init__")

    def forward(self, idx):
        raise AssertionError("TODO: implement TokenEmbedding.forward")


class CausalSelfAttention(nn.Module):
    def __init__(self, embed_dim, num_heads, max_seq_len):
        super().__init__()
        raise AssertionError("TODO: implement CausalSelfAttention.__init__")

    def forward(self, x):
        raise AssertionError("TODO: implement CausalSelfAttention.forward")


class ExpertMLP(nn.Module):
    def __init__(self, embed_dim, mlp_dim):
        super().__init__()
        raise AssertionError("TODO: implement ExpertMLP.__init__")

    def forward(self, x):
        raise AssertionError("TODO: implement ExpertMLP.forward")


class TopKRouter(nn.Module):
    def __init__(self, embed_dim, num_experts, top_k):
        super().__init__()
        raise AssertionError("TODO: implement TopKRouter.__init__")

    def forward(self, x):
        raise AssertionError("TODO: implement TopKRouter.forward")


class SparseMoE(nn.Module):
    def __init__(self, embed_dim, num_experts, top_k, mlp_dim):
        super().__init__()
        raise AssertionError("TODO: implement SparseMoE.__init__")

    def forward(self, x):
        raise AssertionError("TODO: implement SparseMoE.forward")


class MoETransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, num_experts, top_k, mlp_dim, max_seq_len):
        super().__init__()
        raise AssertionError("TODO: implement MoETransformerBlock.__init__")

    def forward(self, x):
        raise AssertionError("TODO: implement MoETransformerBlock.forward")


class MoELM(nn.Module):
    def __init__(self, vocab_size, max_seq_len, embed_dim, depth, num_heads, num_experts, top_k, mlp_dim):
        super().__init__()
        raise AssertionError("TODO: implement MoELM.__init__")

    def forward(self, idx):
        raise AssertionError("TODO: implement MoELM.forward")


def get_dataloaders(num_train, num_val, vocab_size, seq_len, batch_size):
    raise AssertionError("TODO: implement get_dataloaders")


def train_one_epoch(model, loader, optimizer, device):
    raise AssertionError("TODO: implement train_one_epoch")


def evaluate(model, loader, device):
    raise AssertionError("TODO: implement evaluate")


def run_pipeline():
    raise AssertionError("TODO: implement run_pipeline that wires data, model, optimizer, training and evaluation, and returns a dict with train_loss and val_loss")


# =========================
# STUDENT CODE ENDS HERE
# TESTS START BELOW
# =========================


TINY_CONFIG = dict(
    vocab_size=64,
    max_seq_len=16,
    embed_dim=64,
    depth=2,
    num_heads=4,
    num_experts=4,
    top_k=2,
    mlp_dim=128,
)


def _logits_from(model_out):
    return model_out[0] if isinstance(model_out, tuple) else model_out


def test_moe_forward_shape():
    model = MoELM(**TINY_CONFIG)
    idx = torch.randint(0, TINY_CONFIG["vocab_size"], (2, TINY_CONFIG["max_seq_len"]))
    logits = _logits_from(model(idx))
    expected = (2, TINY_CONFIG["max_seq_len"], TINY_CONFIG["vocab_size"])
    assert logits.shape == expected, f"logits shape was {logits.shape}, expected {expected}"
    assert torch.isfinite(logits).all(), "logits contain NaN/inf"


def test_moe_overfits_one_batch():
    torch.manual_seed(0)
    model = MoELM(**TINY_CONFIG)
    idx = torch.randint(0, TINY_CONFIG["vocab_size"], (4, TINY_CONFIG["max_seq_len"]))
    targets = torch.randint(0, TINY_CONFIG["vocab_size"], (4, TINY_CONFIG["max_seq_len"]))
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)
    V = TINY_CONFIG["vocab_size"]

    def ce():
        logits = _logits_from(model(idx))
        return F.cross_entropy(logits.reshape(-1, V), targets.reshape(-1))

    initial = ce().item()
    for _ in range(100):
        optimizer.zero_grad()
        loss = ce()
        loss.backward()
        optimizer.step()
    final = ce().item()
    assert final < 0.5 * initial, f"final loss {final:.4f} should be much lower than initial {initial:.4f}"


def test_pipeline_runs_and_reports_metrics():
    torch.manual_seed(0)
    result = run_pipeline()
    assert isinstance(result, dict), "run_pipeline must return a dict"
    assert "train_loss" in result and "val_loss" in result, "result must contain train_loss and val_loss"


def run_all_tests():
    print("Running MoE pipeline checks...")
    test_moe_forward_shape()
    print("1/3 MoE forward shape passed")
    test_moe_overfits_one_batch()
    print("2/3 model can overfit a single batch")
    test_pipeline_runs_and_reports_metrics()
    print("3/3 end-to-end pipeline returns metrics")
    print("All checks passed. You can mark this assignment done.")


if __name__ == "__main__":
    run_all_tests()
